# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

1. Please take some time to read through the paper and understand their approach, hypotheses and goals.

What was the objective of the study?

- To investigate changes in gene expression in the brain associated with oxycodone dependence and subsequent withdrawal
- To investigate differences in gene expression between mice with and without chronic pain


What do the conditions mean?

- oxy: mouse model receiving chronic oxycodone injection with subsequent withdrawal


- sal: mouse model receiving saline injections as control

What do the genotypes mean?

- SNI: mouse that underwent spared nerve injury (SNI) surgery to model chronic pain


- Sham: mouse that underwent placebo operation as control

Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.

What would you do?

Which groups would you compare to each other?

Please also mention which outcome you would expect to see from each comparison.

1. Analysis:
- quality control of raw and trimmed reads
- adapter trimming
- quantification of gene expression
- differential expression analysis between groups

2. Comparison:
- sal/SNI vs. sal/Sham: changes in gene expression due to chronic pain
- sal/Sham vs. oxy/Sham: changes in gene expression due to oxycodone dependency and withdrawal
- oxy/Sham vs. oxy/SNI: changes in gene expression of oxycodone dependent mice due to chronic pain
- sal/Sham vs oxy/SNI: changes in gene expression of mice with oxycodone dependency and chromic pain to control group without pain and oxycodone withdrawal

Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>
Then, perform some overview analysis and plot the results
1. How many samples do you have per condition?
2. How many samples do you have per genotype?
3. How often do you have each condition per genotype?

In [1]:
import pandas as pd

raw_data = pd.read_excel("conditions_runs_oxy_project.xlsx")
print(raw_data.head())

  Patient          Run RNA-seq  DNA-seq condition: Sal Condition: Oxy  \
0       ?  SRR23195505       x      NaN              x            NaN   
1       ?  SRR23195506       x      NaN            NaN              x   
2       ?  SRR23195507       x      NaN              x            NaN   
3       ?  SRR23195508       x      NaN            NaN              x   
4       ?  SRR23195509       x      NaN            NaN              x   

  Genotype: SNI Genotype: Sham  
0             x            NaN  
1           NaN              x  
2           NaN              x  
3             x            NaN  
4             x            NaN  


In [3]:
# Drop uninformative columns
data = raw_data.drop(columns=["Patient", "RNA-seq", "DNA-seq"])
print(data.head())

           Run condition: Sal Condition: Oxy Genotype: SNI Genotype: Sham
0  SRR23195505              x            NaN             x            NaN
1  SRR23195506            NaN              x           NaN              x
2  SRR23195507              x            NaN           NaN              x
3  SRR23195508            NaN              x             x            NaN
4  SRR23195509            NaN              x             x            NaN


In [17]:
# Replace values with more intuitive once
data = data.fillna(0)
data = data.replace(to_replace="x", value=1)
data = data.rename(columns={"condition: Sal": "Condition: Sal"})
print(data.head())

           Run Condition: Sal Condition: Oxy Genotype: SNI Genotype: Sham
0  SRR23195505              1              0             1              0
1  SRR23195506              0              1             0              1
2  SRR23195507              1              0             0              1
3  SRR23195508              0              1             1              0
4  SRR23195509              0              1             1              0


In [18]:
# 1. Samples per condition:
sal_counts = data["Condition: Sal"].sum()
oxy_counts = data["Condition: Oxy"].sum()

# 2. Samples per genotype:
sni_counts = data["Genotype: SNI"].sum()
sham_counts = data["Genotype: Sham"].sum()

# 3. Number of conditions per genotype:
sal_sni_counts = ((data["Condition: Sal"] == 1) & (data["Genotype: SNI"] == 1)).sum()
sal_sham_counts = ((data["Condition: Sal"] == 1) & (data["Genotype: Sham"] == 1)).sum()
oxy_sni_counts = ((data["Condition: Oxy"] == 1) & (data["Genotype: SNI"] == 1)).sum()
oxy_sham_counts = ((data["Condition: Oxy"] == 1) & (data["Genotype: Sham"] == 1)).sum()

# dataframe:
counts = pd.DataFrame({
    "Category": [
        "Sal",
        "Oxy",
        "SNI",
        "Sham",
        "Sal-SNI",
        "Sal-Sham",
        "Oxy-SNI",
        "Oxy-Sham"
    ],
    "Samples": [
        sal_counts,
        oxy_counts,
        sni_counts,
        sham_counts,
        sal_sni_counts,
        sal_sham_counts,
        oxy_sni_counts,
        oxy_sham_counts
    ]
})

print(counts)

   Category  Samples
0       Sal        8
1       Oxy        8
2       SNI        8
3      Sham        8
4   Sal-SNI        4
5  Sal-Sham        4
6   Oxy-SNI        4
7  Oxy-Sham        4


They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

In [20]:
raw_counts = pd.read_csv("base_counts.csv")
print(raw_counts.head())

           Run       Bases
0  SRR23195505  6922564500
1  SRR23195506  7859530800
2  SRR23195507  8063298900
3  SRR23195508  6927786900
4  SRR23195509  7003550100


In [21]:
data["Base counts"] = raw_counts["Bases"]
data = data.sort_values(by="Base counts")
print(data.head(10))

            Run Condition: Sal Condition: Oxy Genotype: SNI Genotype: Sham  \
11  SRR23195516              0              1             1              0   
6   SRR23195511              0              1             0              1   
12  SRR23195517              0              1             1              0   
0   SRR23195505              1              0             1              0   
3   SRR23195508              0              1             1              0   
14  SRR23195519              0              1             0              1   
4   SRR23195509              0              1             1              0   
9   SRR23195514              0              1             0              1   
5   SRR23195510              1              0             1              0   
7   SRR23195512              1              0             0              1   

    Base counts  
11   6203117700  
6    6456390900  
12   6863840400  
0    6922564500  
3    6927786900  
14   6996050100  
4    7003550100

In [22]:
# Select two smallest runs
smallest_runs = data.nsmallest(2, "Base counts")
print(smallest_runs)

# Create the samplesheet for the pipeline
smallest_runs["Run"].to_csv("ids.csv", index=False, header=False)

            Run Condition: Sal Condition: Oxy Genotype: SNI Genotype: Sham  \
11  SRR23195516              0              1             1              0   
6   SRR23195511              0              1             0              1   

    Base counts  
11   6203117700  
6    6456390900  


In [23]:
# Download the samples with the fetchngs pipeline
!nextflow run nf-core/fetchngs -profile docker --input ids.csv --outdir "results_fetch"
#"/mnt/d/Bioinformatik_MSc_alt/CW_BMD/OUTDIR/Day_2"


 N E X T F L O W   ~  version 26.04.6

Launching `https://github.com/nf-core/fetchngs` [backstabbing_meucci] revision: 955c892d80 [master]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/fetchngs 1.13.0
------------------------------------------------------

Input/output options
  input              : ids.csv
  outdir             : results_fetch

Deprecated options
  trace_report_suffix: 2026-10-05_18-34-39

Core Nextflow options
  revision           : master
  runName            : backstabbing_meucci
  containerEngine    : docker
  launchDir          : /home/laura/CW_BMD/computational-workflows-2026/day_02
  workDir            : /home/laura/CW_BMD/computational-workflows-20

While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.

- identify the sequencing data and experimental groups of the study
- reproduce the conducted workflow by using nf-core pipelines:

    - rnaseq to analyse RNA sequencing data and match the obtained transcripts to a transcriptomic reference producing a gene expression matrix
    
    - differentialabundance to analyse the gene expression matrix and generate differential statistics